In [1]:
!pip -q install feast==0.54.0 pandas pyarrow openpyxl scikit-learn


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.6/7.6 MB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 431.7/431.7 kB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 32.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.9/39.9 MB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.0/212.0 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.3/15.3 MB 57.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 49.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 531.9/531.9 kB 36.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 2.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the sour

In [3]:
import os
import shutil
import subprocess
from pathlib import Path

import pandas as pd
import numpy as np

PROJECT = Path("/content/feast_skillgap")
PROJECT.mkdir(exist_ok=True)

DATASET = Path("/content/CSE_Employability_Dataset.xlsx")

if not DATASET.exists():
    from google.colab import files
    uploaded = files.upload()
    name = next(iter(uploaded))
    shutil.copy2("/content/" + name, DATASET)

print("Ready")


Ready


In [4]:
df = pd.read_excel(DATASET)

print(df.shape)
print(df.columns.tolist())
df.head()


(100, 24)
['StudentID', 'Age', 'Gender', 'CGPA', 'Programming', 'DSA', 'DBMS', 'OS', 'CN', 'Python', 'Java', 'WebDevelopment', 'AI_ML', 'Cloud', 'DevOps', 'CyberSecurity', 'Communication', 'Aptitude', 'Teamwork', 'Internship', 'Certifications', 'Projects', 'PlacementTraining', 'Employability']


,StudentID,Age,Gender,CGPA,Programming,DSA,DBMS,OS,CN,Python,...,DevOps,CyberSecurity,Communication,Aptitude,Teamwork,Internship,Certifications,Projects,PlacementTraining,Employability
0,1,20,Male,8.69,7,7,7,6,6,8,...,7,7,6,7,10,Yes,5,5,Yes,High
1,2,23,Male,7.43,5,4,4,6,5,5,...,3,6,3,5,7,No,0,4,Yes,High
2,3,23,Male,7.87,7,7,6,5,4,6,...,5,4,7,6,8,Yes,5,3,Yes,High
3,4,22,Female,6.40,2,3,1,1,1,1,...,2,1,1,1,7,No,2,1,Yes,Medium
4,5,24,Female,6.41,3,3,4,2,3,2,...,3,2,2,4,5,Yes,0,2,Yes,High


In [5]:
technical_cols = [
    "Programming", "DSA", "DBMS", "OS", "CN", "Python",
    "Java", "WebDevelopment", "AI_ML", "Cloud", "DevOps",
    "CyberSecurity"
]

soft_cols = ["Communication", "Aptitude", "Teamwork"]

features = pd.DataFrame()

features["student_id"] = df["StudentID"].astype("int64")
features["event_timestamp"] = pd.Timestamp("2026-08-01 12:00:00")

features["cgpa"] = df["CGPA"].astype("float32")

for col in technical_cols:
    name = col.lower()
    features[name] = df[col].astype("float32")

for col in soft_cols:
    features[col.lower()] = df[col].astype("float32")

features["certifications"] = df["Certifications"].astype("float32")
features["projects"] = df["Projects"].astype("float32")

features["technical_skill_average"] = (
    df[technical_cols].mean(axis=1).astype("float32")
)

features["soft_skill_average"] = (
    df[soft_cols].mean(axis=1).astype("float32")
)

internship = df["Internship"].eq("Yes").astype(int)
placement = df["PlacementTraining"].eq("Yes").astype(int)

features["experience_score"] = (
    internship + placement + df["Certifications"] + df["Projects"]
).astype("float32")

features["overall_skill_score"] = (
    (features["technical_skill_average"] +
     features["soft_skill_average"]) / 2
).astype("float32")

features.head()


,student_id,event_timestamp,cgpa,programming,dsa,dbms,os,cn,python,java,...,cybersecurity,communication,aptitude,teamwork,certifications,projects,technical_skill_average,soft_skill_average,experience_score,overall_skill_score
0,1,2026-08-01 12:00:00,8.69,7.0,7.0,7.0,6.0,6.0,8.0,7.0,...,7.0,6.0,7.0,10.0,5.0,5.0,6.666667,7.666667,12.0,7.166667
1,2,2026-08-01 12:00:00,7.43,5.0,4.0,4.0,6.0,5.0,5.0,4.0,...,6.0,3.0,5.0,7.0,0.0,4.0,4.500000,5.000000,5.0,4.750000
2,3,2026-08-01 12:00:00,7.87,7.0,7.0,6.0,5.0,4.0,6.0,8.0,...,4.0,7.0,6.0,8.0,5.0,3.0,5.583333,7.000000,10.0,6.291667
3,4,2026-08-01 12:00:00,6.40,2.0,3.0,1.0,1.0,1.0,1.0,2.0,...,1.0,1.0,1.0,7.0,2.0,1.0,1.666667,3.000000,4.0,2.333333
4,5,2026-08-01 12:00:00,6.41,3.0,3.0,4.0,2.0,3.0,2.0,2.0,...,2.0,2.0,4.0,5.0,0.0,2.0,2.916667,3.666667,4.0,3.291667


In [6]:
DATA_DIR = PROJECT / "data"
DATA_DIR.mkdir(exist_ok=True)

PARQUET_FILE = DATA_DIR / "student_features.parquet"

features.to_parquet(PARQUET_FILE, index=False)

print(PARQUET_FILE)


/content/feast_skillgap/data/student_features.parquet


In [7]:
(PROJECT / "feature_store.yaml").write_text('''project: skill_gap_project
registry: data/registry.db
provider: local

online_store:
  path: data/online_store.db

offline_store:
  type: file
''')

print((PROJECT / "feature_store.yaml").read_text())


project: skill_gap_project
registry: data/registry.db
provider: local

online_store:
  path: data/online_store.db

offline_store:
  type: file



In [8]:
(PROJECT / "features.py").write_text('''from datetime import timedelta

from feast import Entity, FeatureView, Field, FileSource
from feast.types import Float32


student = Entity(
    name="student",
    join_keys=["student_id"],
    description="CSE student",
)


student_source = FileSource(
    path="data/student_features.parquet",
    timestamp_field="event_timestamp",
)


student_skill_features = FeatureView(
    name="student_skill_features",
    entities=[student],
    ttl=timedelta(days=3650),
    online=True,
    source=student_source,
    schema=[
        Field(name="cgpa", dtype=Float32),
        Field(name="programming", dtype=Float32),
        Field(name="dsa", dtype=Float32),
        Field(name="dbms", dtype=Float32),
        Field(name="os", dtype=Float32),
        Field(name="cn", dtype=Float32),
        Field(name="python", dtype=Float32),
        Field(name="java", dtype=Float32),
        Field(name="webdevelopment", dtype=Float32),
        Field(name="ai_ml", dtype=Float32),
        Field(name="cloud", dtype=Float32),
        Field(name="devops", dtype=Float32),
        Field(name="cybersecurity", dtype=Float32),
        Field(name="communication", dtype=Float32),
        Field(name="aptitude", dtype=Float32),
        Field(name="teamwork", dtype=Float32),
        Field(name="certifications", dtype=Float32),
        Field(name="projects", dtype=Float32),
        Field(name="technical_skill_average", dtype=Float32),
        Field(name="soft_skill_average", dtype=Float32),
        Field(name="experience_score", dtype=Float32),
        Field(name="overall_skill_score", dtype=Float32),
    ],
)
''')

print((PROJECT / "features.py").read_text())


from datetime import timedelta

from feast import Entity, FeatureView, Field, FileSource
from feast.types import Float32


student = Entity(
    name="student",
    join_keys=["student_id"],
    description="CSE student",
)


student_source = FileSource(
    path="data/student_features.parquet",
    timestamp_field="event_timestamp",
)


student_skill_features = FeatureView(
    name="student_skill_features",
    entities=[student],
    ttl=timedelta(days=3650),
    online=True,
    source=student_source,
    schema=[
        Field(name="cgpa", dtype=Float32),
        Field(name="programming", dtype=Float32),
        Field(name="dsa", dtype=Float32),
        Field(name="dbms", dtype=Float32),
        Field(name="os", dtype=Float32),
        Field(name="cn", dtype=Float32),
        Field(name="python", dtype=Float32),
        Field(name="java", dtype=Float32),
        Field(name="webdevelopment", dtype=Float32),
        Field(name="ai_ml", dtype=Float32),
        Field(name="cloud", dty

In [9]:
result = subprocess.run(
    ["feast", "apply"],
    cwd=PROJECT,
    capture_output=True,
    text=True
)

print(result.stdout)

if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError("feast apply failed")


No project found in the repository. Using project name skill_gap_project defined in feature_store.yaml
Applying changes for project skill_gap_project
Created project skill_gap_project
Created entity student
Created feature view student_skill_features

Created sqlite table skill_gap_project_student_skill_features




In [10]:
from feast import FeatureStore

store = FeatureStore(repo_path=str(PROJECT))

entity_df = df[["StudentID"]].copy()
entity_df.columns = ["student_id"]
entity_df["event_timestamp"] = pd.Timestamp("2026-08-01 12:00:00")

feature_names = [
    "cgpa",
    "programming",
    "dsa",
    "dbms",
    "os",
    "cn",
    "python",
    "java",
    "webdevelopment",
    "ai_ml",
    "cloud",
    "devops",
    "cybersecurity",
    "communication",
    "aptitude",
    "teamwork",
    "certifications",
    "projects",
    "technical_skill_average",
    "soft_skill_average",
    "experience_score",
    "overall_skill_score",
]

feature_refs = [
    f"student_skill_features:{name}"
    for name in feature_names
]

historical_features = store.get_historical_features(
    entity_df=entity_df,
    features=feature_refs
).to_df()

historical_features.head()


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,student_id,event_timestamp,cgpa,programming,dsa,dbms,os,cn,python,java,...,cybersecurity,communication,aptitude,teamwork,certifications,projects,technical_skill_average,soft_skill_average,experience_score,overall_skill_score
0,1,2026-08-01 12:00:00+00:00,8.69,7.0,7.0,7.0,6.0,6.0,8.0,7.0,...,7.0,6.0,7.0,10.0,5.0,5.0,6.666667,7.666667,12.0,7.166667
1,73,2026-08-01 12:00:00+00:00,6.87,3.0,5.0,4.0,3.0,3.0,2.0,3.0,...,2.0,2.0,2.0,7.0,1.0,5.0,3.583333,3.666667,7.0,3.625000
2,72,2026-08-01 12:00:00+00:00,8.78,8.0,8.0,7.0,9.0,6.0,7.0,8.0,...,9.0,8.0,7.0,8.0,0.0,3.0,7.583333,7.666667,5.0,7.625000
3,71,2026-08-01 12:00:00+00:00,6.83,3.0,2.0,3.0,4.0,4.0,3.0,3.0,...,3.0,1.0,4.0,7.0,5.0,5.0,2.916667,4.000000,12.0,3.458333
4,70,2026-08-01 12:00:00+00:00,8.35,8.0,6.0,8.0,6.0,6.0,7.0,9.0,...,7.0,6.0,8.0,9.0,2.0,4.0,7.000000,7.666667,8.0,7.333333


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [11]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report

model_data = historical_features.merge(
    df[["StudentID", "Employability"]],
    left_on="student_id",
    right_on="StudentID",
    how="left"
)

X = model_data[feature_names]
y_text = model_data["Employability"]

encoder = LabelEncoder()
y = encoder.fit_transform(y_text)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    class_weight="balanced"
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)

print("Model Accuracy:", accuracy)
print("Model Accuracy (%):", round(accuracy * 100, 2))
print(classification_report(
    y_test,
    y_pred,
    target_names=encoder.classes_,
    zero_division=0
))


Model Accuracy: 0.95
Model Accuracy (%): 95.0
              precision    recall  f1-score   support

        High       1.00      1.00      1.00        15
         Low       0.00      0.00      0.00         1
      Medium       0.80      1.00      0.89         4

    accuracy                           0.95        20
   macro avg       0.60      0.67      0.63        20
weighted avg       0.91      0.95      0.93        20



/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [12]:
store.materialize(
    start_date=pd.Timestamp("2026-07-31").to_pydatetime(),
    end_date=pd.Timestamp("2026-08-02").to_pydatetime()
)

print("Materialization completed")


Materializing 1 feature views from 2026-07-31 00:00:00+00:00 to 2026-08-02 00:00:00+00:00 into the sqlite online store.

student_skill_features:
Materialization completed


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [13]:
student_id = 1

online_result = store.get_online_features(
    features=feature_refs,
    entity_rows=[{"student_id": student_id}]
).to_dict()

online_values = {
    key: value[0]
    for key, value in online_result.items()
}

online_values


{'student_id': 1,
 'os': 6.0,
 'devops': 7.0,
 'soft_skill_average': 7.666666507720947,
 'cn': 6.0,
 'cybersecurity': 7.0,
 'dbms': 7.0,
 'java': 7.0,
 'experience_score': 12.0,
 'teamwork': 10.0,
 'technical_skill_average': 6.666666507720947,
 'python': 8.0,
 'overall_skill_score': 7.166666507720947,
 'cloud': 6.0,
 'communication': 6.0,
 'aptitude': 7.0,
 'webdevelopment': 6.0,
 'ai_ml': 6.0,
 'cgpa': 8.6899995803833,
 'projects': 5.0,
 'certifications': 5.0,
 'programming': 7.0,
 'dsa': 7.0}

In [17]:
online_row = pd.DataFrame([{
    feature: online_values[feature]
    for feature in feature_names
}])

prediction_number = model.predict(online_row)[0]

final_prediction = encoder.inverse_transform(
    [prediction_number]
)[0]

print("StudentID:", student_id)
print("Final Prediction:", final_prediction)

StudentID: 1
Final Prediction: High


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [18]:
print("===== RESULTS TO PUT IN README =====")
print()
print("Historical Feature Output:")
print(
    historical_features[
        historical_features["student_id"] == 1
    ][[
        "student_id",
        "cgpa",
        "technical_skill_average",
        "soft_skill_average",
        "experience_score",
        "overall_skill_score"
    ]]
)
print()
print("Model Accuracy:", round(accuracy * 100, 2), "%")
print()
print("Online Feature Output:")
for key, value in online_values.items():
    print(key, "=", value)
print()
print("Final Prediction:", final_prediction)


===== RESULTS TO PUT IN README =====

Historical Feature Output:
   student_id  cgpa  technical_skill_average  soft_skill_average  \
0           1  8.69                 6.666667            7.666667   

   experience_score  overall_skill_score  
0              12.0             7.166667  

Model Accuracy: 95.0 %

Online Feature Output:
student_id = 1
os = 6.0
devops = 7.0
soft_skill_average = 7.666666507720947
cn = 6.0
cybersecurity = 7.0
dbms = 7.0
java = 7.0
experience_score = 12.0
teamwork = 10.0
technical_skill_average = 6.666666507720947
python = 8.0
overall_skill_score = 7.166666507720947
cloud = 6.0
communication = 6.0
aptitude = 7.0
webdevelopment = 6.0
ai_ml = 6.0
cgpa = 8.6899995803833
projects = 5.0
certifications = 5.0
programming = 7.0
dsa = 7.0

Final Prediction: High


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
